In [1]:
import numpy as np

class Board():
    def __init__(self, n=3, m=3, k=None):
        self.n = n  # filas
        self.m = m  # columnas
        self.k = k if k is not None else min(n, m)  # fichas en línea para ganar
        self.state = np.zeros((n, m))

    def valid_moves(self):
        return [(i, j) for i in range(self.n) for j in range(self.m) if self.state[i, j] == 0]

    def update(self, symbol, row, col):
        if self.state[row, col] == 0:
            self.state[row, col] = symbol
        else:
            raise ValueError("movimiento ilegal!")

    def check_winner(self, symbol):
        k = self.k
        # filas
        for i in range(self.n):
            for j in range(self.m - k + 1):
                if all(self.state[i, j+l] == symbol for l in range(k)):
                    return True
        # columnas
        for i in range(self.n - k + 1):
            for j in range(self.m):
                if all(self.state[i+l, j] == symbol for l in range(k)):
                    return True
        # diagonal principal
        for i in range(self.n - k + 1):
            for j in range(self.m - k + 1):
                if all(self.state[i+l, j+l] == symbol for l in range(k)):
                    return True
        # diagonal inversa
        for i in range(self.n - k + 1):
            for j in range(k - 1, self.m):
                if all(self.state[i+l, j-l] == symbol for l in range(k)):
                    return True
        return False

    def is_game_over(self):
        if self.check_winner(1):
            return 1
        if self.check_winner(-1):
            return -1
        if len(self.valid_moves()) == 0:
            return 0
        return None

    def reset(self):
        self.state = np.zeros((self.n, self.m))

In [2]:
class Agent():
    def __init__(self, alpha=0.5, prob_exp=0.5):
        self.value_function = {}
        self.alpha = alpha
        self.positions = []
        self.prob_exp = prob_exp

    def reset(self):
        self.positions = []

    def move(self, board, explore=True):
        valid_moves = board.valid_moves()
        if explore and np.random.uniform(0, 1) < self.prob_exp:
            ix = np.random.choice(len(valid_moves))
            return valid_moves[ix]
        max_value = -1000
        for row, col in valid_moves:
            next_board = board.state.copy()
            next_board[row, col] = self.symbol
            next_state = str(next_board.reshape(board.n * board.m))  # <-- n*m
            value = 0 if self.value_function.get(next_state) is None else self.value_function.get(next_state)
            if value >= max_value:
                max_value = value
                best_row, best_col = row, col
        return best_row, best_col

    def update(self, board):
        self.positions.append(str(board.state.reshape(board.n * board.m)))  # <-- n*m

    def reward(self, reward):
        for p in reversed(self.positions):
            if self.value_function.get(p) is None:
                self.value_function[p] = 0
            self.value_function[p] += self.alpha * (reward - self.value_function[p])
            reward = self.value_function[p]
            

In [3]:
from tqdm import tqdm

class Game():
    def __init__(self, player1, player2, n=3, m=3, k=None):
        player1.symbol = 1
        player2.symbol = -1
        self.players = [player1, player2]
        self.board = Board(n=n, m=m, k=k)

    def selfplay(self, rounds=100):
        wins = [0, 0]
        for i in tqdm(range(1, rounds + 1)):
            self.board.reset()
            for player in self.players:
                player.reset()

            players_this_round = self.players[:]
            np.random.shuffle(players_this_round)

            game_over = False
            while not game_over:
                for player in players_this_round:
                    action = player.move(self.board)
                    self.board.update(player.symbol, action[0], action[1])
                    for player in self.players:
                        player.update(self.board)
                    if self.board.is_game_over() is not None:
                        game_over = True
                        break
            self.reward()
            for ix, player in enumerate(self.players):
                if self.board.is_game_over() == player.symbol:
                    wins[ix] += 1
        return wins

    def reward(self):
        winner = self.board.is_game_over()
        if winner == 0:
            for player in self.players:
                player.reward(0.5)
        else:
            for player in self.players:
                if winner == player.symbol:
                    player.reward(1)
                else:
                    player.reward(0)

In [ ]:
# 3x3 clásico, gana con 3 en línea
game = Game(Agent(), Agent(), n=3, m=3)
game.selfplay(300000)

# 3x4, gana con 3 en línea
game = Game(Agent(), Agent(), n=3, m=4, k=3)
game.selfplay(400000)

# 4x4, gana con 4 en línea
game = Game(Agent(), Agent(), n=4, m=4, k=4)
game.selfplay(500000)

# conecta 4: tablero 6x7, gana con 4 en línea
game = Game(Agent(), Agent(), n=6, m=7, k=4)
game.selfplay(1000000)

In [4]:
class AgentOptimista():
    def __init__(self, alpha=0.5, optimistic_value=1.0):
        self.value_function = {}
        self.alpha = alpha
        self.optimistic_value = optimistic_value
        self.positions = []

    def reset(self):
        self.positions = []

    def get_value(self, state, board):
        key = str(state.reshape(board.n * board.m))
        if key not in self.value_function:
            self.value_function[key] = self.optimistic_value  # desconocido -> optimista
        return self.value_function[key]

    def move(self, board, explore=False):
        valid_moves = board.valid_moves()
        best_value = -1000
        best_actions = []
        for row, col in valid_moves:
            next_board = board.state.copy()
            next_board[row, col] = self.symbol
            value = self.get_value(next_board, board)
            if value > best_value:
                best_value = value
                best_actions = [(row, col)]
            elif value == best_value:
                best_actions.append((row, col))  # desempate aleatorio
        ix = np.random.choice(len(best_actions))
        return best_actions[ix]

    def update(self, board):
        self.positions.append(str(board.state.reshape(board.n * board.m)))

    def reward(self, reward):
        for p in reversed(self.positions):
            if p not in self.value_function:
                self.value_function[p] = self.optimistic_value  # <-- optimista en vez de 0
            self.value_function[p] += self.alpha * (reward - self.value_function[p])
            reward = self.value_function[p]

In [5]:
class AgentGradiente():
    def __init__(self, alpha=0.1):
        self.preferences = {}
        self.alpha = alpha
        self.positions = []
        self.pi_history = []
        self.recompensas = []

    def reset(self):
        self.positions = []
        self.pi_history = []

    def get_preference(self, state, board):
        key = str(state.reshape(board.n * board.m))
        return self.preferences.get(key, 0.0)

    def softmax(self, H):
        H = np.array(H)
        exp_H = np.exp(H - H.max())  # estabilidad numérica
        return exp_H / exp_H.sum()

    def move(self, board, explore=True):
        valid_moves = board.valid_moves()
        H = []
        for row, col in valid_moves:
            next_board = board.state.copy()
            next_board[row, col] = self.symbol
            H.append(self.get_preference(next_board, board))

        pi = self.softmax(H)

        if explore:
            ix = np.random.choice(len(valid_moves), p=pi)
            self.pi_history.append((pi, ix))
        else:
            ix = np.argmax(pi)  # inferencia: el más probable

        return valid_moves[ix]

    def update(self, board):
        self.positions.append(str(board.state.reshape(board.n * board.m)))

    def reward(self, reward):
        self.recompensas.append(reward)
        recompensa_media = np.mean(self.recompensas)

        # positions[0] es el estado inicial sin acción
        for ix, p in enumerate(self.positions[1:]):
            if p not in self.preferences:
                self.preferences[p] = 0.0
            pi, chosen = self.pi_history[ix]
            for j in range(len(pi)):
                if j == chosen:
                    self.preferences[p] += self.alpha * (reward - recompensa_media) * (1 - pi[j])
                else:
                    self.preferences[p] -= self.alpha * (reward - recompensa_media) * pi[j]

In [ ]:
# 3x3 con valores optimistas
agent1 = AgentOptimista(alpha=0.5, optimistic_value=1.0)
agent2 = AgentOptimista(alpha=0.5, optimistic_value=1.0)
game = Game(agent1, agent2, n=3, m=3)
game.selfplay(300000)

# 3x4 con gradiente
agent1 = AgentGradiente(alpha=0.1)
agent2 = AgentGradiente(alpha=0.1)
game = Game(agent1, agent2, n=3, m=4, k=3)
game.selfplay(400000)

# mezclar agentes
agent1 = AgentOptimista(alpha=0.5, optimistic_value=1.0)
agent2 = AgentGradiente(alpha=0.1)
game = Game(agent1, agent2, n=3, m=3)
game.selfplay(300000)

In [6]:
class AgentEgreedy():
    def __init__(self, alpha=0.5, prob_exp=0.5):
        self.value_function = {}
        self.alpha = alpha
        self.prob_exp = prob_exp  # epsilon
        self.positions = []

    def reset(self):
        self.positions = []

    def move(self, board, explore=True):
        valid_moves = board.valid_moves()

        # EXPLORACIÓN: con probabilidad epsilon -> acción aleatoria
        if explore and np.random.uniform(0, 1) < self.prob_exp:
            ix = np.random.choice(len(valid_moves))
            return valid_moves[ix]

        # EXPLOTACIÓN: elegir el estado con mayor valor conocido
        max_value = -1000
        best_actions = []
        for row, col in valid_moves:
            next_board = board.state.copy()
            next_board[row, col] = self.symbol
            next_state = str(next_board.reshape(board.n * board.m))
            value = self.value_function.get(next_state, 0)
            if value > max_value:
                max_value = value
                best_actions = [(row, col)]
            elif value == max_value:
                best_actions.append((row, col))  # desempate aleatorio

        ix = np.random.choice(len(best_actions))
        return best_actions[ix]

    def update(self, board):
        self.positions.append(str(board.state.reshape(board.n * board.m)))

    def reward(self, reward):
        for p in reversed(self.positions):
            if p not in self.value_function:
                self.value_function[p] = 0
            self.value_function[p] += self.alpha * (reward - self.value_function[p])
            reward = self.value_function[p]

In [ ]:
from tqdm import tqdm
import matplotlib.pyplot as plt

def entrenar_y_registrar(agent1, agent2, n=3, m=3, k=None, rounds=100000, intervalo=1000):
    """entrena y registra victorias cada 'intervalo' rondas"""
    game = Game(agent1, agent2, n=n, m=m, k=k)
    historial = []  # victorias acumuladas de agent1

    total_wins = [0, 0]
    for i in tqdm(range(1, rounds + 1)):
        game.board.reset()
        for player in game.players:
            player.reset()

        players_this_round = game.players[:]
        np.random.shuffle(players_this_round)

        game_over = False
        while not game_over:
            for player in players_this_round:
                action = player.move(game.board)
                game.board.update(player.symbol, action[0], action[1])
                for p in game.players:
                    p.update(game.board)
                if game.board.is_game_over() is not None:
                    game_over = True
                    break
        game.reward()
        for ix, player in enumerate(game.players):
            if game.board.is_game_over() == player.symbol:
                total_wins[ix] += 1

        if i % intervalo == 0:
            historial.append(total_wins[0] / i)  # tasa de victoria de agent1

    return historial


# entrenar los tres métodos enfrentados contra el mismo rival (AgentEgreedy base)
rounds = 100000
intervalo = 1000
x = list(range(intervalo, rounds + 1, intervalo))

# epsilon alto -> mucha exploración
rival1 = AgentEgreedy(prob_exp=0.5)
egreedy_alto = AgentEgreedy(alpha=0.5, prob_exp=0.5)
hist_egreedy_alto = entrenar_y_registrar(egreedy_alto, rival1, rounds=rounds, intervalo=intervalo)

# epsilon bajo -> poca exploración
rival2 = AgentEgreedy(prob_exp=0.5)
egreedy_bajo = AgentEgreedy(alpha=0.5, prob_exp=0.1)
hist_egreedy_bajo = entrenar_y_registrar(egreedy_bajo, rival2, rounds=rounds, intervalo=intervalo)

# valores optimistas
rival3 = AgentEgreedy(prob_exp=0.5)
optimista = AgentOptimista(alpha=0.5, optimistic_value=1.0)
hist_optimista = entrenar_y_registrar(optimista, rival3, rounds=rounds, intervalo=intervalo)

# gradiente
rival4 = AgentEgreedy(prob_exp=0.5)
gradiente = AgentGradiente(alpha=0.1)
hist_gradiente = entrenar_y_registrar(gradiente, rival4, rounds=rounds, intervalo=intervalo)

# graficar
plt.figure(figsize=(10, 5))
plt.plot(x, hist_egreedy_alto,  label='ε-greedy ε=0.5')
plt.plot(x, hist_egreedy_bajo,  label='ε-greedy ε=0.1')
plt.plot(x, hist_optimista,     label='Valores optimistas')
plt.plot(x, hist_gradiente,     label='Gradiente')
plt.axhline(y=0.5, color='gray', linestyle='--', label='50% (empate teórico)')
plt.xlabel('Partidas')
plt.ylabel('Tasa de victoria')
plt.title('Comparación de estrategias de exploración (3x3)')
plt.legend()
plt.grid(True)
plt.show()